# 07 — Tipos y Arquitecturas de Agentes de IA

> **Bloque:** LLMs | **Nivel:** Avanzado
>
> Complementario al tutorial [07-tipos-agentes.md](../../llms/07-tipos-agentes.md)

Este notebook implementa y compara los principales tipos de agentes:
1. Agente reactivo vs deliberativo
2. Agente reflexivo (Self-Critique)
3. Arquitecturas multi-agente (pipeline y jerárquico)
4. Sistemas de memoria
5. Patrones de planificación: ReAct vs Plan & Execute
6. Evaluación comparativa

In [ ]:
# %pip install anthropic python-dotenv matplotlib

import anthropic
import json
import time
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from datetime import datetime
from collections import deque

%matplotlib inline

client = anthropic.Anthropic()
MODELO_RAPIDO  = 'claude-haiku-4-5-20251001'
MODELO_POTENTE = 'claude-sonnet-4-6'
print('✓ Listo')

## 1. Agente reactivo vs deliberativo

Comparamos la misma tarea con dos enfoques distintos.

In [ ]:
def agente_reactivo(texto: str) -> dict:
    """Responde directamente sin planificar."""
    t0 = time.time()
    r = client.messages.create(
        model=MODELO_RAPIDO, max_tokens=256, temperature=0.0,
        system='Clasifica el sentimiento: POSITIVO, NEGATIVO o NEUTRO. Solo la etiqueta.',
        messages=[{'role': 'user', 'content': texto}]
    )
    return {
        'tipo': 'reactivo',
        'respuesta': r.content[0].text.strip(),
        'tiempo': round(time.time() - t0, 2),
        'tokens': r.usage.input_tokens + r.usage.output_tokens
    }

def agente_deliberativo(texto: str) -> dict:
    """Razona antes de responder."""
    t0 = time.time()
    r = client.messages.create(
        model=MODELO_POTENTE, max_tokens=512, temperature=0.0,
        system="""Analiza el texto paso a paso antes de clasificar:
1. ¿Qué palabras clave indican emoción?
2. ¿El contexto es ambiguo?
3. Clasificación final: POSITIVO, NEGATIVO o NEUTRO

Responde en JSON: {"razonamiento": "...", "clasificacion": "...", "confianza": 0.0-1.0}""",
        messages=[{'role': 'user', 'content': texto}]
    )
    try:
        datos = json.loads(r.content[0].text)
    except Exception:
        datos = {'clasificacion': r.content[0].text, 'confianza': 0.5}
    datos.update({
        'tipo': 'deliberativo',
        'tiempo': round(time.time() - t0, 2),
        'tokens': r.usage.input_tokens + r.usage.output_tokens
    })
    return datos

# Textos ambiguos para comparar
textos = [
    'El producto llegó rápido pero la calidad es mejorable',        # Ambiguo
    'Increíble servicio, lo recomiendo sin dudarlo',                # Claramente positivo
    'No esperaba gran cosa y... tampoco me sorprendió para bien',   # Ambiguo/negativo
]

print(f'{"Texto":<50} {"Reactivo":<15} {"Deliberativo":<15} {"Confianza"}')
print('─' * 95)
for texto in textos:
    r = agente_reactivo(texto)
    d = agente_deliberativo(texto)
    confianza = d.get('confianza', '?')
    print(f'{texto[:48]:<50} {r["respuesta"]:<15} {d.get("clasificacion", "?"):<15} {confianza}')

print(f'\nTokens reactivo (último): {r["tokens"]} | Deliberativo: {d["tokens"]}')

## 2. Agente reflexivo (Self-Critique)

El agente genera una respuesta, la critica y la mejora.

In [ ]:
def agente_reflexivo(tarea: str, max_iter: int = 2, verbose: bool = True) -> dict:
    """Agente Actor + Crítico que itera hasta aprobación."""
    historial_actor = [{'role': 'user', 'content': tarea}]
    versiones = []

    for i in range(max_iter):
        # Actor genera o mejora
        r_actor = client.messages.create(
            model=MODELO_POTENTE, max_tokens=800,
            system='Eres el Actor. Genera la mejor respuesta posible.',
            messages=historial_actor
        )
        respuesta = r_actor.content[0].text
        versiones.append(respuesta)

        if verbose:
            print(f'[v{i+1}] {respuesta[:120]}...\n')

        # Crítico evalúa
        r_critico = client.messages.create(
            model=MODELO_POTENTE, max_tokens=300,
            system='"""Eres el Crítico. Evalúa la respuesta.\nSi es buena: responde exactamente APROBADO\nSi no: responde MEJORAR: [instrucciones concretas]"""',
            messages=[{'role': 'user', 'content': f'Tarea: {tarea}\n\nRespuesta:\n{respuesta}'}]
        )
        feedback = r_critico.content[0].text

        if verbose:
            print(f'Crítico: {feedback[:80]}\n')

        if 'APROBADO' in feedback:
            if verbose: print(f'✅ Aprobado en iteración {i+1}')
            return {'respuesta': respuesta, 'iteraciones': i+1, 'versiones': versiones, 'aprobado': True}

        historial_actor.append({'role': 'assistant', 'content': respuesta})
        historial_actor.append({'role': 'user', 'content': f'Feedback:\n{feedback}\n\nMejora tu respuesta.'})

    return {'respuesta': respuesta, 'iteraciones': max_iter, 'versiones': versiones, 'aprobado': False}


resultado = agente_reflexivo(
    'Explica qué es el overfitting en machine learning y cómo evitarlo',
    max_iter=2
)
print(f'\n📊 Estadísticas: {resultado["iteraciones"]} iteración(es) | Aprobado: {resultado["aprobado"]}')

## 3. Arquitectura jerárquica (Orquestador + Subagentes)

In [ ]:
def subagente(tarea: str, especialidad: str) -> str:
    """Subagente especializado."""
    r = client.messages.create(
        model=MODELO_RAPIDO, max_tokens=600,
        system=f'Eres un experto en {especialidad}. Responde de forma concisa y precisa.',
        messages=[{'role': 'user', 'content': tarea}]
    )
    return r.content[0].text

def orquestador(objetivo: str, verbose: bool = True) -> str:
    """Descompone el objetivo y coordina subagentes en paralelo."""
    # Paso 1: Planificar subtareas
    r_plan = client.messages.create(
        model=MODELO_POTENTE, max_tokens=512,
        messages=[{'role': 'user', 'content': f"""
Descompón en 3 subtareas independientes (para agentes especializados en paralelo).
Objetivo: {objetivo}
Responde SOLO con JSON: [{{"subtarea": "...", "especialidad": "..."}}]
"""}]
    )
    subtareas = json.loads(r_plan.content[0].text)

    if verbose:
        print(f'📋 Plan: {len(subtareas)} subtareas')
        for st in subtareas:
            print(f'  → [{st["especialidad"]}] {st["subtarea"][:60]}...')

    # Paso 2: Ejecutar en paralelo
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=len(subtareas)) as ex:
        futuros = {ex.submit(subagente, st['subtarea'], st['especialidad']): st for st in subtareas}
        resultados = {st['especialidad']: f.result() for f, st in futuros.items()}
    tiempo_paralelo = round(time.time() - t0, 1)

    if verbose:
        print(f'\n⚡ Subagentes completados en {tiempo_paralelo}s (paralelo)')

    # Paso 3: Síntesis
    contexto = '\n\n'.join(f'=== {esp} ===\n{res}' for esp, res in resultados.items())
    r_final = client.messages.create(
        model=MODELO_POTENTE, max_tokens=1200,
        messages=[{'role': 'user', 'content': f"""
Objetivo: {objetivo}

Resultados de subagentes especializados:
{contexto}

Sintetiza en una respuesta coherente, integrada y completa.
"""}]
    )
    return r_final.content[0].text


print('=== Agente jerárquico ===' )
resultado = orquestador(
    'Analiza los riesgos, oportunidades y regulación de la IA en el sector financiero europeo'
)
print('\n=== SÍNTESIS FINAL ===')
print(resultado[:600] + '...')

## 4. Sistemas de memoria

In [ ]:
# ── Memoria comprimida ────────────────────────────────────────────────────────

class AgenteMemoriaComprimida:
    """Comprime el historial cuando supera cierto tamaño."""

    def __init__(self, max_mensajes: int = 6):
        self.historial = []
        self.resumen = ''
        self.max_mensajes = max_mensajes
        self.compresiones = 0

    def _comprimir(self):
        texto = '\n'.join(
            f"{m['role'].upper()}: {m['content']}" for m in self.historial[:-2]
        )
        r = client.messages.create(
            model=MODELO_RAPIDO, max_tokens=300,
            messages=[{'role': 'user', 'content': f'Resume en ≤150 palabras:\n{texto}'}]
        )
        self.resumen = (self.resumen + '\n' + r.content[0].text).strip()
        self.historial = self.historial[-2:]
        self.compresiones += 1
        print(f'  📦 Compresión #{self.compresiones} aplicada')

    def chatear(self, mensaje: str) -> str:
        if len(self.historial) >= self.max_mensajes:
            self._comprimir()

        self.historial.append({'role': 'user', 'content': mensaje})
        system = 'Eres un asistente con memoria de la conversación.'
        if self.resumen:
            system += f'\n\nContexto previo comprimido:\n{self.resumen}'

        r = client.messages.create(
            model=MODELO_RAPIDO, max_tokens=300, system=system,
            messages=self.historial
        )
        respuesta = r.content[0].text
        self.historial.append({'role': 'assistant', 'content': respuesta})
        return respuesta


print('=== Demostración de memoria comprimida ===')
agente = AgenteMemoriaComprimida(max_mensajes=4)

conversacion = [
    'Hola, mi nombre es Ana y trabajo en análisis de datos',
    'Estoy aprendiendo sobre agentes de IA para mi trabajo',
    '¿Cuáles son los tipos principales de agentes?',
    '¿Qué tipo de agente me recomendarías para analizar datos de ventas?',
    '¿Recuerdas cómo me llamo y en qué trabajo?',  # Test de memoria
]

for turno, msg in enumerate(conversacion, 1):
    print(f'\nUsuario ({turno}): {msg}')
    respuesta = agente.chatear(msg)
    print(f'IA: {respuesta[:150]}...' if len(respuesta) > 150 else f'IA: {respuesta}')

print(f'\n📊 Mensajes en historial activo: {len(agente.historial)}')
print(f'📊 Compresiones realizadas: {agente.compresiones}')
print(f'📊 Resumen acumulado: {len(agente.resumen)} chars')

## 5. Plan & Execute vs ReAct

In [ ]:
# Herramientas compartidas
TOOLS_DEF = [
    {'name': 'calcular',
     'description': 'Evalúa expresiones matemáticas',
     'input_schema': {'type': 'object', 'properties': {'expresion': {'type': 'string'}}, 'required': ['expresion']}},
    {'name': 'obtener_fecha',
     'description': 'Devuelve la fecha actual',
     'input_schema': {'type': 'object', 'properties': {}}},
    {'name': 'buscar_dato',
     'description': 'Busca datos en una base de conocimiento',
     'input_schema': {'type': 'object', 'properties': {'clave': {'type': 'string'}}, 'required': ['clave']}},
]

BASE_DATOS = {
    'precio_euro': 1.08,
    'empleados_empresa': 150,
    'ventas_2025': 12_000_000,
    'año_fundacion': 2010,
}

def ejecutar_herramienta(nombre: str, args: dict) -> dict:
    if nombre == 'calcular':
        try:
            return {'resultado': eval(args['expresion'], {'__builtins__': {}}, {'math': math})}
        except Exception as e:
            return {'error': str(e)}
    elif nombre == 'obtener_fecha':
        return {'fecha': datetime.now().strftime('%d/%m/%Y'), 'año': datetime.now().year}
    elif nombre == 'buscar_dato':
        valor = BASE_DATOS.get(args['clave'])
        return {'clave': args['clave'], 'valor': valor or 'No encontrado'}
    return {'error': f'Herramienta {nombre} no encontrada'}


# ── ReAct ──────────────────────────────────────────────────────────────────────
def react(objetivo: str, max_pasos: int = 6) -> dict:
    mensajes = [{'role': 'user', 'content': objetivo}]
    pasos = 0
    t0 = time.time()

    while pasos < max_pasos:
        pasos += 1
        r = client.messages.create(
            model=MODELO_POTENTE, max_tokens=1024, tools=TOOLS_DEF,
            system='Usa las herramientas disponibles para resolver la tarea paso a paso.',
            messages=mensajes
        )
        mensajes.append({'role': 'assistant', 'content': r.content})

        if r.stop_reason == 'end_turn':
            texto = next((b.text for b in r.content if hasattr(b, 'text')), '')
            return {'patron': 'ReAct', 'respuesta': texto, 'pasos': pasos,
                    'tiempo': round(time.time()-t0, 1)}

        resultados = []
        for b in r.content:
            if b.type == 'tool_use':
                res = ejecutar_herramienta(b.name, b.input)
                resultados.append({'type': 'tool_result', 'tool_use_id': b.id,
                                   'content': json.dumps(res)})
        mensajes.append({'role': 'user', 'content': resultados})

    return {'patron': 'ReAct', 'respuesta': 'Límite de pasos', 'pasos': pasos,
            'tiempo': round(time.time()-t0, 1)}


# ── Plan & Execute ─────────────────────────────────────────────────────────────
def plan_execute(objetivo: str) -> dict:
    t0 = time.time()
    herramientas_disponibles = ['calcular', 'obtener_fecha', 'buscar_dato']

    # Fase 1: Plan
    r_plan = client.messages.create(
        model=MODELO_POTENTE, max_tokens=512,
        messages=[{'role': 'user', 'content': f"""
Objetivo: {objetivo}
Herramientas: {herramientas_disponibles}

Genera un plan en JSON:
{{"pasos": [{{"herramienta": "nombre", "args": {{}}, "descripcion": "..."}}]}}
"""}]
    )
    plan = json.loads(r_plan.content[0].text)
    pasos_plan = plan.get('pasos', [])
    print(f'  Plan generado: {len(pasos_plan)} paso(s)')

    # Fase 2: Ejecución
    resultados = []
    for paso in pasos_plan:
        res = ejecutar_herramienta(paso['herramienta'], paso.get('args', {}))
        resultados.append({'paso': paso['descripcion'], 'resultado': res})

    # Fase 3: Síntesis
    r_final = client.messages.create(
        model=MODELO_POTENTE, max_tokens=512,
        messages=[{'role': 'user', 'content': f"""
Objetivo: {objetivo}
Resultados: {json.dumps(resultados, ensure_ascii=False)}
Genera la respuesta final.
"""}]
    )
    return {'patron': 'Plan&Execute', 'respuesta': r_final.content[0].text,
            'pasos': len(pasos_plan)+2, 'tiempo': round(time.time()-t0, 1)}


# Comparación
objetivo_test = 'Calcula cuántos años tiene la empresa (fundada en 2010) y cuánto es el 15% de las ventas de 2025'
print(f'Objetivo: {objetivo_test}\n')

print('--- ReAct ---')
r_react = react(objetivo_test)
print(f'Respuesta: {r_react["respuesta"][:200]}')

print('\n--- Plan & Execute ---')
r_pe = plan_execute(objetivo_test)
print(f'Respuesta: {r_pe["respuesta"][:200]}')

print(f'\n📊 ReAct: {r_react["pasos"]} pasos, {r_react["tiempo"]}s')
print(f'📊 Plan&Execute: {r_pe["pasos"]} pasos, {r_pe["tiempo"]}s')

## 6. Visualización comparativa de patrones

In [ ]:
# Comparativa cualitativa de patrones y arquitecturas
patrones = [
    {'nombre': 'Reactivo', 'velocidad': 9, 'calidad': 4, 'coste': 1, 'complejidad': 1, 'memoria': 0},
    {'nombre': 'ReAct', 'velocidad': 6, 'calidad': 7, 'coste': 5, 'complejidad': 5, 'memoria': 3},
    {'nombre': 'Plan\n& Execute', 'velocidad': 5, 'calidad': 8, 'coste': 6, 'complejidad': 6, 'memoria': 2},
    {'nombre': 'Reflexión', 'velocidad': 3, 'calidad': 9, 'coste': 8, 'complejidad': 7, 'memoria': 2},
    {'nombre': 'Jerárquico', 'velocidad': 7, 'calidad': 9, 'coste': 9, 'complejidad': 9, 'memoria': 4},
    {'nombre': 'BDI', 'velocidad': 4, 'calidad': 8, 'coste': 7, 'complejidad': 9, 'memoria': 7},
]

dims = ['velocidad', 'calidad', 'coste', 'complejidad', 'memoria']
etiquetas = ['Velocidad', 'Calidad', 'Coste', 'Complejidad\nimpl.', 'Capacidad\nmemoria']
colores = ['#3498DB', '#2ECC71', '#E74C3C', '#9B59B6', '#F39C12', '#1ABC9C']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for idx, (patron, ax, color) in enumerate(zip(patrones, axes, colores)):
    valores = [patron[d] for d in dims]
    bars = ax.bar(etiquetas, valores, color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.set_ylim(0, 10)
    ax.set_title(patron['nombre'], fontsize=11, fontweight='bold', color=color)
    ax.set_yticks(range(0, 11, 2))
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', labelsize=8)

    for bar, val in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                str(val), ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Comparativa de patrones y arquitecturas de agentes', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Escala 0-10: mayor es mejor (excepto Coste y Complejidad donde menor = más eficiente)')

## 7. Árbol de decisión: ¿qué agente usar?

In [ ]:
def recomendar_agente(descripcion_tarea: str) -> dict:
    """Usa el LLM para recomendar el tipo de agente más adecuado."""
    r = client.messages.create(
        model=MODELO_POTENTE, max_tokens=600,
        system="""Eres un arquitecto de sistemas de IA. Analiza la descripción de una tarea
y recomienda el tipo de agente más adecuado.

Tipos disponibles:
- REACTIVO: tarea simple, sin estado, respuesta rápida
- DELIBERATIVO: requiere planificación y modelo del mundo
- REACT: exploración iterativa con herramientas
- PLAN_EXECUTE: plan revisable antes de ejecutar
- REFLEXION: calidad crítica, autocorrección
- JERARQUICO: múltiples subtareas especializadas en paralelo
- BDI: estado complejo con creencias dinámicas
- SWARM: múltiples agentes simples colaborando

Responde en JSON:
{
  "tipo": "NOMBRE",
  "razon": "explicación de 2-3 frases",
  "modelo_recomendado": "haiku|sonnet|opus",
  "necesita_memoria": true|false,
  "alternativa": "otro tipo posible si el primero es complejo"
}""",
        messages=[{'role': 'user', 'content': f'Tarea: {descripcion_tarea}'}]
    )
    return json.loads(r.content[0].text)


tareas_ejemplo = [
    'Clasificar automáticamente los emails entrantes en categorías',
    'Investigar y redactar un informe sobre tendencias del mercado de IA',
    'Asistente de soporte que recuerda el historial de cada cliente',
    'Sistema que analiza código, ejecuta tests y propone correcciones',
    'Coordinar 5 agentes especializados para analizar un contrato legal',
]

print(f'{"Tarea":<55} {"Tipo":<18} {"Modelo":<10} {"Memoria"}')
print('─' * 95)
for tarea in tareas_ejemplo:
    rec = recomendar_agente(tarea)
    print(f'{tarea[:53]:<55} {rec["tipo"]:<18} {rec["modelo_recomendado"]:<10} {"Sí" if rec["necesita_memoria"] else "No"}')
    print(f'  → {rec["razon"][:90]}...')
    if rec.get('alternativa'):
        print(f'  → Alternativa: {rec["alternativa"]}')
    print()

## 8. Tu turno — diseña tu propio agente

In [ ]:
# Define tu propia tarea y obtén una recomendación + implementación básica
mi_tarea = """
Describe aquí tu caso de uso...
"""

if len(mi_tarea.split()) > 5:
    # Recomendación de tipo de agente
    rec = recomendar_agente(mi_tarea)
    print(f'Tipo recomendado: {rec["tipo"]}')
    print(f'Modelo: {rec["modelo_recomendado"]}')
    print(f'Razón: {rec["razon"]}')
    print(f'Necesita memoria: {rec["necesita_memoria"]}')
    if rec.get('alternativa'):
        print(f'Alternativa más simple: {rec["alternativa"]}')

    # Generar esqueleto de implementación
    r = client.messages.create(
        model=MODELO_POTENTE, max_tokens=1000,
        messages=[{'role': 'user', 'content': f"""
Tarea: {mi_tarea}
Tipo de agente elegido: {rec['tipo']}

Genera un esqueleto Python básico (sin código de API externo, solo estructura)
para implementar este tipo de agente. Incluye comentarios explicativos.
"""}]
    )
    print(f'\n=== Esqueleto de implementación ===')
    print(r.content[0].text)
else:
    print('Modifica mi_tarea con tu propio caso de uso para obtener una recomendación personalizada.')

---
**Tutorial relacionado:** [07-tipos-agentes.md](../../llms/07-tipos-agentes.md)